In [0]:
##Carregando os dados

df = spark.table("workspace.default.vr_education")
display(df)

student_id,age,gender,grade_level,field_of_study,usage_of_vr_in_education,hours_of_vr_usage_per_week,engagement_level,improvement_in_learning_outcomes,subject,instructor_vr_proficiency,perceived_effectiveness_of_vr,access_to_vr_equipment,impact_on_creativity,stress_level_with_vr_usage,collaboration_with_peers_via_vr,feedback_from_educators_on_vr,interest_in_continuing_vr_based_learning,region,school_support_for_vr_in_curriculum
STUD0001,13,Non-binary,Postgraduate,Science,No,6,1,Yes,Computer Science,Intermediate,3,Yes,5,High,No,Neutral,No,South America,No
STUD0002,16,Non-binary,Undergraduate,Medicine,No,6,1,Yes,Math,Beginner,2,Yes,3,Low,Yes,Positive,No,Oceania,No
STUD0003,15,Prefer not to say,High School,Science,No,4,5,Yes,Art,Advanced,5,Yes,2,Low,Yes,Neutral,Yes,Oceania,No
STUD0004,24,Female,Postgraduate,Engineering,Yes,2,4,No,Economics,Beginner,5,No,3,High,No,Neutral,No,Europe,Yes
STUD0005,22,Non-binary,Undergraduate,Arts,Yes,10,3,No,Art,Beginner,4,Yes,1,Medium,No,Negative,Yes,North America,Yes
STUD0006,28,Male,High School,Science,Yes,10,5,No,Economics,Intermediate,1,Yes,1,Low,No,Neutral,No,Asia,Yes
STUD0007,19,Male,Undergraduate,Business,Yes,9,4,Yes,History,Intermediate,4,Yes,5,Medium,Yes,Neutral,No,North America,Yes
STUD0008,19,Male,High School,Education,No,1,5,No,Physics,Beginner,5,Yes,2,Low,No,Neutral,Yes,Oceania,No
STUD0009,29,Non-binary,Undergraduate,Law,Yes,0,2,Yes,Computer Science,Beginner,2,No,2,High,Yes,Positive,Yes,Asia,No
STUD0010,16,Prefer not to say,Postgraduate,Engineering,Yes,10,5,Yes,Art,Beginner,4,No,1,Medium,Yes,Positive,Yes,Africa,No


In [0]:
##Padronizar nomes das colunas
from pyspark.sql.functions import col

df = df.toDF(*[c.lower() for c in df.columns])

In [0]:
##Vendo valores nulos
from pyspark.sql.functions import isnan, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

student_id,age,gender,grade_level,field_of_study,usage_of_vr_in_education,hours_of_vr_usage_per_week,engagement_level,improvement_in_learning_outcomes,subject,instructor_vr_proficiency,perceived_effectiveness_of_vr,access_to_vr_equipment,impact_on_creativity,stress_level_with_vr_usage,collaboration_with_peers_via_vr,feedback_from_educators_on_vr,interest_in_continuing_vr_based_learning,region,school_support_for_vr_in_curriculum
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
##Padronizando colunas categóricas

from pyspark.sql.functions import lower, trim

columns_to_clean = [
    "gender",
    "region",
    "usage_of_vr_in_education",
    "stress_level_with_vr_usage",
    "impact_on_creativity",
    "access_to_vr_equipment",
    "collaboration_with_peers_via_vr" #Corrigido aqui
]

for c in columns_to_clean:
    df = df.withColumn(c, lower(trim(col(c))))

In [0]:
#Verificando valores únicos (antes e depois)
df.select("gender").distinct().show()
df.select("region").distinct().show()
df.select("engagement_level").distinct().show()
df.select("perceived_effectiveness_of_vr").distinct().show()

+-----------------+
|           gender|
+-----------------+
|       non-binary|
|prefer not to say|
|           female|
|             male|
+-----------------+

+-------------+
|       region|
+-------------+
|south america|
|      oceania|
|       europe|
|north america|
|         asia|
|       africa|
+-------------+

+----------------+
|engagement_level|
+----------------+
|               1|
|               5|
|               4|
|               3|
|               2|
+----------------+

+-----------------------------+
|perceived_effectiveness_of_vr|
+-----------------------------+
|                            3|
|                            2|
|                            5|
|                            4|
|                            1|
+-----------------------------+



In [0]:
#Removendo duplicatas
df = df.dropDuplicates()

In [0]:
##Verificando tipos
df.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- grade_level: string (nullable = true)
 |-- field_of_study: string (nullable = true)
 |-- usage_of_vr_in_education: string (nullable = true)
 |-- hours_of_vr_usage_per_week: long (nullable = true)
 |-- engagement_level: long (nullable = true)
 |-- improvement_in_learning_outcomes: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- instructor_vr_proficiency: string (nullable = true)
 |-- perceived_effectiveness_of_vr: long (nullable = true)
 |-- access_to_vr_equipment: string (nullable = true)
 |-- impact_on_creativity: string (nullable = true)
 |-- stress_level_with_vr_usage: string (nullable = true)
 |-- collaboration_with_peers_via_vr: string (nullable = true)
 |-- feedback_from_educators_on_vr: string (nullable = true)
 |-- interest_in_continuing_vr_based_learning: string (nullable = true)
 |-- region: string (nullable = true)
 |-- school_suppor

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("engagement_level", col("engagement_level").cast("int"))
df = df.withColumn("perceived_effectiveness_of_vr", col("perceived_effectiveness_of_vr").cast("int"))

## Conclusão da Limpeza de Dados

O dataset foi tratado e padronizado com sucesso:

- Não foram encontrados valores nulos relevantes
- Colunas categóricas foram padronizadas (lower + trim)
- Não há duplicidade de categorias
- Dados prontos para análise exploratória

O dataset está preparado para a etapa de análise (EDA).

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.default.vr_education_clean")